# Лабораторна робота №4: Розробка AI-асистента продуктового аналітика

Роботу виконала студентка групи ІП-34 Бараневич Еліна.

У цій роботі розглядається створення практичного інструменту для продуктового аналітика. Основна ідея полягає в тому, щоб прибрати рутину первинного аналізу, але не довіряти моделі самостійні розрахунки. Усі математичні операції, підрахунки частот, пошук викидів та сегментація виконуються класичним кодом за допомогою бібліотек Pandas та NumPy. Мовна модель використовується виключно для того, щоб якісно інтерпретувати вже пораховані цифри, виявляти логічні суперечності та формулювати висновки зрозумілою для бізнесу мовою.

Важливо зазначити особливість структури інструменту. Перша функція первинного аудиту даних є абсолютно універсальною. Вона може обробити будь-який табличний файл незалежно від тематики, чи це демографія, логістика, чи медичні записи. Проте наступні три функції орієнтовані на продуктову аналітику та SaaS-бізнес. Їхня логіка націлена на роботу з такими характеристиками, як підписки, канали залучення, користувацька активність, життєвий цикл та повторні замовлення, щоб отримати практичні висновки саме для цієї предметної області.


In [ ]:
!pip install google-genai

In [ ]:
import json
import numpy as np
import pandas as pd
import os
from google import genai
from google.colab import userdata
from google.genai import types
import pprint

### Отримання вихідних даних через Kaggle API
Для дослідження використовується вибірка поведінки клієнтів. Щоб не завантажувати файл вручну на комп'ютер, ми налаштовуємо пряме завантаження через офіційний інтерфейс Kaggle. Програма зчитує секретний токен KAGGLE_API_TOKEN, який заздалегідь збережено у вкладці Secrets у Colab, і виконує завантаження архіву безпосередньо у віртуальну машину. Після розархівування за допомогою функції read_csv з бібліотеки Pandas файл зчитується у змінну df, і на екран виводяться його розміри та перші три рядки для швидкого візуального контролю структури. Якщо ви захочете використати власний локальний файл, цей блок можна замінити на стандартний виклик read_csv із шляхом до вашого документа.

In [2]:
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle datasets download -d bhaskerpaul/sales-and-marketing-dataset --unzip
csv_files = [f for f in os.listdir(".") if f.endswith(".csv")]
print("Знайдені CSV файли:", csv_files)
df = pd.read_csv(csv_files[0])
print("\nРозмір датасету:", df.shape)
display(df.head(3))

Dataset URL: https://www.kaggle.com/datasets/bhaskerpaul/sales-and-marketing-dataset
License(s): apache-2.0
100% 1.12M/1.12M [00:00<00:00, 116MB/s]

Знайдені CSV файли: ['Sales - Marketing customer dataset.csv']

Розмір датасету: (15000, 30)


,customer_id,gender,age,country,city,signup_date,last_purchase_date,acquisition_channel,device_type,subscription_type,...,support_tickets,refund_requested,delivery_delay_days,payment_method,satisfaction_score,nps_score,marketing_spend_per_user,lifetime_value,last_3_month_purchase_freq,churn
0,10001,Male,52.0,India,Berlin,2022-05-10 00:00:00,2024-12-31 00:00:00,Email,Tablet,Annual,...,0,0,3,UPI,3.0,10,27.56,915.310827,14,0
1,10002,NaN,35.0,Germany,Mumbai,2024-06-16 00:00:00,2024-05-07 00:00:00,Organic,Desktop,Monthly,...,5,0,3,BKash,3.0,7,15.15,2079.960938,11,0
2,10003,Female,27.0,Germany,London,2023-08-23 00:00:00,2024-04-28 00:00:00,Email,Mobile,Annual,...,1,0,2,UPI,5.0,6,13.51,1379.150885,9,0


## Завдання 1. Автоматичний аудит якості даних
Цей блок реалізує повне первинне дослідження датасету. Логіка коду повністю універсальна і проходить по кожній колонці таблиці незалежно від її назви. Для кожного стовпця код за допомогою методів notnull та isnull підраховує кількість та частку пропущених значень, а також кількість унікальних записів через nunique. За допомогою евристик код перевіряє, чи не є текстове поле насправді прихованою датою або забрудненим числом, з якого потрібно прибрати символи валют.

Для числових полів обчислюються основні описові статистики: мінімум, максимум, середнє значення, медіана та стандартне відхилення. Також реалізовано пошук викидів за правилом міжквартильного розмаху, де відсікаються значення, що виходять за межі півтора міжквартильних діапазонів. Для колонок із невеликою кількістю унікальних значень формується їхній повний частотний розподіл за допомогою value_counts. Крім того, у загальний словник додається невеликий зріз перших реальних рядків таблиці. Це необхідно для того, щоб мовна модель могла оцінити узгодженість даних між різними полями в межах одного запису, наприклад, чи відповідає вказане місто країні і чи не передує покупка реєстрації.

Після формування структурованого словника функція підставляє його в обраний варіант інструкції для моделі Gemini. Якщо обрано третю версію промпту, модель повертає форматований JSON, який додаткова функція перетворює на зв'язний, зручний для читання людиною аналітичний звіт з оцінкою загального стану даних та кроками для їх виправлення.

In [3]:
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

def generate_eda_profile(df, max_categories=30, sample_size=5):
    total_rows, total_cols = len(df), len(df.columns)
    if total_rows == 0: return {"error": "Датасет порожній"}
    profile = {
        "dataset_summary": {"total_rows": total_rows, "total_columns": total_cols, "total_duplicates": int(df.duplicated().sum()), "duplicate_percentage": round(float(df.duplicated().mean() * 100), 2)},
        "sample_records": df.head(sample_size).to_dict(orient="records"),
        "columns_analysis": {}
    }
    for col in df.columns:
        series = df[col]
        non_null_cnt, null_cnt = int(series.notnull().sum()), int(series.isnull().sum())
        n_unique = int(series.nunique(dropna=True))
        col_data = {"declared_type": str(series.dtype), "missing_values": {"count": null_cnt, "percentage": round(float(null_cnt / total_rows * 100), 2)}, "unique_values_count": n_unique, "detected_nature": "unknown"}
        is_datetime = pd.api.types.is_datetime64_any_dtype(series)
        parsed_dates = None
        if not is_datetime and (series.dtype == 'object' or str(series.dtype) == 'string'):
            sample = series.dropna().head(100).astype(str)
            if any(k in col.lower() for k in ["date", "time", "created", "timestamp"]) or sample.str.contains(r'^\d{4}[-/.]\d{2}[-/.]\d{2}').mean() > 0.5:
                parsed_dates = pd.to_datetime(series, errors='coerce')
                if parsed_dates.notnull().mean() > 0.6: is_datetime = True
        if is_datetime:
            col_data["detected_nature"] = "datetime"
            dates_series = series if parsed_dates is None else parsed_dates
            valid_dates = dates_series.dropna()
            inv_cnt = int(total_rows - null_cnt - len(valid_dates))
            col_data["datetime_stats"] = {"min_date": str(valid_dates.min()) if not valid_dates.empty else None, "max_date": str(valid_dates.max()) if not valid_dates.empty else None, "invalid_date_formats_count": inv_cnt}
            profile["columns_analysis"][col] = col_data
            continue
        is_numeric = pd.api.types.is_numeric_dtype(series)
        numeric_series = None
        if not is_numeric and (series.dtype == 'object' or str(series.dtype) == 'string'):
            cleaned = series.astype(str).str.replace(r'[$,€£]', '', regex=True).str.strip()
            converted = pd.to_numeric(cleaned, errors='coerce')
            if non_null_cnt > 0 and (converted.notnull().sum() / non_null_cnt) > 0.75:
                is_numeric = True
                numeric_series = converted
                col_data["detected_nature"] = "numeric (dirty string format)"
        if is_numeric:
            num = series.dropna() if numeric_series is None else numeric_series.dropna()
            if col_data["detected_nature"] == "unknown": col_data["detected_nature"] = "numeric"
            if not num.empty:
                q1, median, q3 = float(num.quantile(0.25)), float(num.quantile(0.50)), float(num.quantile(0.75))
                iqr = q3 - q1
                outliers = int(((num < (q1 - 1.5 * iqr)) | (num > (q3 + 1.5 * iqr))).sum())
                col_data["numeric_stats"] = {"min": float(num.min()), "max": float(num.max()), "mean": round(float(num.mean()), 2), "median": round(median, 2), "outliers_iqr_count": outliers, "negative_values_count": int((num < 0).sum())}
            profile["columns_analysis"][col] = col_data
            continue
        if n_unique <= max_categories:
            col_data["detected_nature"] = "categorical"
            cats = {}
            for k, v in series.value_counts(dropna=False).head(max_categories).items(): cats[str(k)] = int(v)
            col_data["category_distribution"] = cats
        else:
            col_data["detected_nature"] = "high-cardinality identifier / text"
            samples = {}
            for k, v in series.value_counts(dropna=True).head(3).items(): samples[str(k)] = int(v)
            col_data["sample_top_values"] = samples
        profile["columns_analysis"][col] = col_data
    return profile

EDA_PROMPTS = {
    "v1": """Ти аналітик даних. Подивись на цей JSON зі статистикою та семплом рядків і напиши звіт про якість даних.
Які колонки брудні, де є аномалії, які колонки суперечать одна одній та що треба почистити?
Дані:
{profile_json}""",

    "v2": """Ти провідний продуктовий аналітик. Проведи аудит якості даних на основі технічного профілю та прикладів рядків.
Вимоги:
1. Посилайся на точні цифри з JSON (пропуски, викиди, аномалії).
2. Зістав стовпці між собою за прикладами з sample_records (наприклад, відповідність географії, логіки статусів тощо).
3. Напиши структурований звіт для людей:
   - Огляд стану датасету
   - Невідповідності між колонками та артефакти значень
   - Технічні проблеми з типами та пропусками
   - Покроковий план очищення
Дані:
{profile_json}""",

    "v3": """<context>
Ти — провідний дата-аналітик. Твоя мета — підготувати детальний, легко читабельний текстовий звіт про аудит якості даних для команди розробки та аналітики.
</context>
<rules>
1. Посилайся ТІЛЬКИ на конкретні числа з <dataset_profile>. Не вигадуй метрик.
2. Обов'язково проаналізуй блок <sample_records>: порівняй значення у взаємопов'язаних колонках (наприклад, країна vs місто, дати реєстрації vs покупок) на предмет логічних суперечностей.
3. Поверни структурований JSON, у якому текстові поля вже сформульовані зрозумілою людською мовою у вигляді зв'язних абзаців.
</rules>
<dataset_profile>
{profile_json}
</dataset_profile>
<instructions>
Сформуй JSON строго наступної структури:
{{
  "health_score": 85,
  "executive_summary": "Зв'язний абзац (4-5 речень) простою мовою про стан датасету, ключові ризики та придатність до аналітики.",
  "cross_column_issues": [
    {{"columns_involved": ["Колонка1", "Колонка2"], "description": "Детальний опис виявленої суперечності на основі sample_records."}}
  ],
  "column_level_issues": [
    {{"column": "Назва колонки", "issue_description": "Розгорнуте пояснення проблеми з точними цифрами.", "business_impact": "Чим це загрожує бізнес-висновкам."}}
  ],
  "action_plan_text": "Зв'язний покроковий текст із чіткими рекомендаціями для дата-інженера щодо очищення даних."
}}
</instructions>"""
}

def format_v3_to_human_text(data):
    if not isinstance(data, dict) or "health_score" not in data: return str(data)
    lines = [f"ОЦІНКА ЗДОРОВ'Я ДАНИХ: {data.get('health_score')}/100\n", "ЗАГАЛЬНИЙ ВИСНОВОК:", f"{data.get('executive_summary', '')}\n"]
    if data.get("cross_column_issues"):
        lines.append(" СУПЕРЕЧНОСТІ МІЖ СТОВПЦЯМИ (ЗНАЙДЕНО В СЕМПЛАХ):")
        for it in data["cross_column_issues"]: lines.append(f" • [{', '.join(it.get('columns_involved', []))}]: {it.get('description')}")
        lines.append("")
    if data.get("column_level_issues"):
        lines.append("ПРОБЛЕМИ ОКРЕМИХ СТОВПЦІВ:")
        for it in data["column_level_issues"]:
            lines.append(f" • Стовпець '{it.get('column')}': {it.get('issue_description')}")
            lines.append(f"   Вплив на аналітику: {it.get('business_impact')}")
        lines.append("")
    lines.append("ПОКРОКОВИЙ ПЛАН ОЧИЩЕННЯ:")
    lines.append(data.get("action_plan_text", ""))
    return "\n".join(lines)

def run_eda(df, prompt_version="v3", model="gemini-2.5-flash"):
    profile = generate_eda_profile(df, sample_size=5)
    p_json = json.dumps(profile, ensure_ascii=False, indent=2)
    prompt = EDA_PROMPTS[prompt_version].format(profile_json=p_json)
    cfg = types.GenerateContentConfig(temperature=0.1, response_mime_type="application/json" if prompt_version == "v3" else "text/plain")
    res = client.models.generate_content(model=model, contents=prompt, config=cfg).text.strip()
    if prompt_version == "v3":
        try: return format_v3_to_human_text(json.loads(res))
        except: return res
    return res

## Механізм визначення схеми даних для аналітики SaaS-бізнесу
Оскільки датасети в різних компаніях мають власні назви колонок, жорстко прив'язувати код до конкретних імен не можна. Для того щоб наступні аналітичні функції могли автоматично адаптуватися до будь-якої таблиці у сфері SaaS або підпискових сервісів, тут реалізовано автоматичне розмічення схеми.

Логіка коду аналізує наявні поля таблиці за характерними ключовими словами та типами значень. Вона шукає головну числову метрику бізнесу серед полів, пов'язаних із виручкою, чеками чи сумами витрат. Також вона ідентифікує стовпчики з датами, перетворюючи їх на часовий формат Pandas. Серед текстових колонок із невеликою кількістю унікальних значень виділяються основні розрізи для сегментації, наприклад канали залучення, типи пристроїв або тарифні плани, а також виявляються прапорці знижок та числові оцінки задоволеності. Таким чином формується динамічна карта полів, завдяки якій подальші розрахунки залишаються повністю універсальними.

In [4]:
def map_dataframe_schema(df):
    clean_df = df.copy()
    cols = list(clean_df.columns)
    mapping = {"metric_col": None, "date_cols": [], "segments": [], "rating_col": None, "binary_or_coupon_col": None}

    for c in cols:
        if clean_df[c].dtype == 'object' or str(clean_df[c].dtype) == 'string':
            cleaned_num = pd.to_numeric(clean_df[c].astype(str).str.replace(r'[$,€£]', '', regex=True).str.strip(), errors='coerce')
            if clean_df[c].notnull().sum() > 0 and (cleaned_num.notnull().sum() / clean_df[c].notnull().sum()) > 0.75:
                clean_df[c] = cleaned_num

    metric_keywords = ['spent', 'revenue', 'amount', 'sales', 'price', 'cost', 'total', 'val', 'income']
    for kw in metric_keywords:
        for c in cols:
            if kw in c.lower() and pd.api.types.is_numeric_dtype(clean_df[c]):
                mapping["metric_col"] = c
                break
        if mapping["metric_col"]: break
    if not mapping["metric_col"]:
        num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(clean_df[c])]
        if num_cols: mapping["metric_col"] = num_cols[0]

    date_keywords = ['date', 'time', 'timestamp', 'created', 'signup', 'purchase', 'period']
    for c in cols:
        is_dt = pd.api.types.is_datetime64_any_dtype(clean_df[c])
        if not is_dt and (clean_df[c].dtype == 'object' or str(clean_df[c].dtype) == 'string'):
            if any(k in c.lower() for k in date_keywords):
                parsed = pd.to_datetime(clean_df[c], errors='coerce')
                if parsed.notnull().mean() > 0.5:
                    clean_df[c] = parsed
                    is_dt = True
        if is_dt: mapping["date_cols"].append(c)

    for c in cols:
        if c != mapping["metric_col"] and c not in mapping["date_cols"]:
            n_unq = clean_df[c].nunique(dropna=True)
            if 2 <= n_unq <= 35:
                mapping["segments"].append(c)
                if any(k in c.lower() for k in ['coupon', 'promo', 'discount', 'flag', 'is_', 'has_']):
                    mapping["binary_or_coupon_col"] = c

    for c in cols:
        if any(k in c.lower() for k in ['score', 'rating', 'sat', 'feedback', 'grade']) and pd.api.types.is_numeric_dtype(clean_df[c]):
            mapping["rating_col"] = c
            break

    return (clean_df, mapping)

## Завдання 2. Генерація перевірюваних продуктових гіпотез
У цьому розділі реалізовано генерацію гіпотез для пошуку точок росту бізнесу. На першому етапі числовий код розраховує реальні розриви в показниках між різними сегментами клієнтів. За допомогою групування groupby для кожного виявленого сегмента обчислюється кількість користувачів, середні та медіанні витрати. Якщо в даних присутній фактор використання промокодів або знижок, код окремо рахує різницю у витратах між групами, які купували за повною ціною та зі знижкою. Якщо є дві дати, обчислюється реальна тривалість життєвого циклу клієнтів у днях.

Усі ці точні розрахунки пакуються в окремий документ і надсилаються до моделі Gemini. Завдання штучного інтелекту полягає не в бізнес-інтерпретації отриманих розривів. Модель формулює гіпотези у суворому форматі: конкретне спостереження на основі наших цифр, припущення щодо поведінкової причини та чітка дія або експеримент, який аналітик має провести для перевірки.

In [5]:
def extract_hypotheses_features(df):
    clean_df, mapping = map_dataframe_schema(df)
    m_col = mapping["metric_col"]
    if not m_col: return {"error": "Не знайдено числової метрики для розрахунку гіпотез"}

    summary = {"target_metric": m_col, "segments_performance": {}}

    for seg in mapping["segments"]:
        seg_stats = {}
        for cat, group in clean_df.groupby(seg):
            vals = group[m_col].dropna()
            if not vals.empty:
                seg_stats[str(cat)] = {"count": int(len(group)), "mean": round(float(vals.mean()), 2), "median": round(float(vals.median()), 2)}
        summary["segments_performance"][seg] = seg_stats

    bin_col = mapping["binary_or_coupon_col"]
    if bin_col:
        top_val = clean_df[bin_col].dropna().value_counts().index[0]
        has_flag = clean_df[bin_col].notnull() & (clean_df[bin_col] == top_val)
        summary["feature_impact"] = {
            "feature_name": bin_col,
            "group_primary": {"label": str(top_val), "count": int(has_flag.sum()), "mean_metric": round(float(clean_df[has_flag][m_col].dropna().mean()), 2)},
            "group_other": {"label": "Other/None", "count": int((~has_flag).sum()), "mean_metric": round(float(clean_df[~has_flag][m_col].dropna().mean()), 2)}
        }

    if len(mapping["date_cols"]) >= 2:
        d1, d2 = mapping["date_cols"][0], mapping["date_cols"][1]
        diff_days = (clean_df[d2] - clean_df[d1]).dt.days.abs().dropna()
        summary["lifecycle_days"] = {"between_dates": f"{d1} and {d2}", "mean_days": round(float(diff_days.mean()), 1), "median_days": round(float(diff_days.median()), 1)}

    r_col = mapping["rating_col"]
    if r_col:
        rat_stats = {}
        for r_val, group in clean_df.groupby(r_col):
            vals = group[m_col].dropna()
            if not vals.empty: rat_stats[str(r_val)] = {"count": int(len(group)), "mean_metric": round(float(vals.mean()), 2)}
        summary["rating_vs_metric"] = {"rating_column": r_col, "breakdown": rat_stats}

    return summary

HYPOTHESES_PROMPTS = {
    "v1": """Ти продуктовий аналітик. Подивись на агреговані бізнес-показники з датасету та згенеруй 5-10 гіпотез, які варто перевірити команді.
Дані:
{metrics_json}""",

    "v2": """Ти провідний продуктовий аналітик. Твоє завдання — сформулювати від 5 до 8 продуктових та маркетингових гіпотез на основі розрахунків.
Правила:
1. Кожна гіпотеза повинна явно містити точні цифри з наданого JSON (розриви у виручці, кількості користувачів тощо). Жодних голослівних тверджень.
2. Формат кожної гіпотези: Спостереження (з цифрами) -> Припущення (чому так) -> Що перевірити далі.
Показники:
{metrics_json}""",

    "v3": """<context>
Ти — Lead Growth / Product Analyst. Твоя мета — згенерувати перевірювані продуктові гіпотези на базі розрахованих сегментних метрик для оптимізації продукту та маркетингу.
</context>
<rules>
1. Сформуй від 5 до 8 гіпотез строго на основі <calculated_metrics>.
2. Обов'язкова прив'язка до фактів: кожна гіпотеза має посилатися на конкретні числа.
3. Відповідь має бути строго у валідному JSON, тексти всередині мають бути написані професійною мовою.
</rules>
<calculated_metrics>
{metrics_json}
</calculated_metrics>
<instructions>
Поверни JSON наступної структури:
{{
  "hypotheses_overview": "Короткий вступний абзац про знайдені точки росту та розриви в сегментах.",
  "hypotheses_list": [
    {{
      "id": 1,
      "title": "Коротка назва гіпотези",
      "data_fact": "Конкретний числовий розрив або аномалія з наданих метрик",
      "rationale": "Чому так відбувається (бізнес-причина)",
      "validation_action": "Який експеримент або тест треба провести аналітику"
    }}
  ]
}}
</instructions>"""
}

def format_hypotheses_v3(data):
    if not isinstance(data, dict) or "hypotheses_list" not in data: return str(data)
    lines = ["ПРОДУКТОВІ ГІПОТЕЗИ ДЛЯ ПЕРЕВІРКИ:\n", f"{data.get('hypotheses_overview', '')}\n"]
    for h in data["hypotheses_list"]:
        lines.append(f" Гіпотеза №{h.get('id')}: {h.get('title')}")
        lines.append(f" • Факт із даних: {h.get('data_fact')}")
        lines.append(f" • Припущення: {h.get('rationale')}")
        lines.append(f" • Як перевірити: {h.get('validation_action')}\n")
    return "\n".join(lines)

def run_hypotheses(df, prompt_version="v3", model="gemini-2.5-flash"):
    metrics = extract_hypotheses_features(df)
    p_json = json.dumps(metrics, ensure_ascii=False, indent=2)
    prompt = HYPOTHESES_PROMPTS[prompt_version].format(metrics_json=p_json)
    cfg = types.GenerateContentConfig(temperature=0.2, response_mime_type="application/json" if prompt_version == "v3" else "text/plain")
    res = client.models.generate_content(model=model, contents=prompt, config=cfg).text.strip()
    if prompt_version == "v3":
        try: return format_hypotheses_v3(json.loads(res))
        except: return res
    return res

## Завдання 4. Пояснення аномалій та факторний аналіз просідання
Цей блок присвячено розслідуванню різких змін у ключових показниках бізнесу. Код бере знайдену часову колонку та розбиває всю історію транзакцій на місячні інтервали за допомогою періодів Pandas. Потім він порівнює два останні місяці між собою, обчислюючи абсолютне та відсоткове відхилення загальної виручки.

Далі запускається факторний аналіз: сумарна різниця між періодами декомпонується за всіма доступними зрізами, такими як канали залучення, пристрої, стать чи географія. За кожною категорією вираховується її точний внесок у зміну загального показника, і сегменти сортуються від найбільшого падіння до найменшого. Модель Gemini отримує цю декомпозицію і визначає головні сегменти-винуватці просідання, формулює гіпотези технічного чи бізнес-характеру та пропонує невідкладні заходи для команди.

In [6]:
def extract_anomaly_breakdown(df):
    clean_df, mapping = map_dataframe_schema(df)
    m_col = mapping["metric_col"]
    d_col = mapping["date_cols"][-1] if mapping["date_cols"] else None

    if not m_col or not d_col:
        return {"error": f"Для факторного аналізу потрібні числова метрика та дата. Виявлено: metric={m_col}, date={d_col}"}

    valid_df = clean_df.dropna(subset=[m_col, d_col]).copy()
    if len(valid_df) == 0: return {"error": "Недостатньо заповнених даних для порівняння періодів"}

    valid_df['period'] = valid_df[d_col].dt.to_period('M')
    periods = sorted(valid_df['period'].unique())
    if len(periods) < 2: return {"error": "Для декомпозиції потрібно щонайменше 2 періоди дат"}

    p_prev, p_curr = periods[-2], periods[-1]
    df_prev, df_curr = valid_df[valid_df['period'] == p_prev], valid_df[valid_df['period'] == p_curr]
    v_prev, v_curr = float(df_prev[m_col].sum()), float(df_curr[m_col].sum())
    total_delta = v_curr - v_prev
    pct_change = round((total_delta / v_prev * 100), 2) if v_prev > 0 else 0.0

    breakdown = {}
    for seg in mapping["segments"][:4]:
        prev_grp = df_prev.groupby(seg)[m_col].sum()
        curr_grp = df_curr.groupby(seg)[m_col].sum()
        seg_diffs = []
        for cat in set(prev_grp.index).union(set(curr_grp.index)):
            vp = float(prev_grp.get(cat, 0.0))
            vc = float(curr_grp.get(cat, 0.0))
            diff = vc - vp
            seg_diffs.append({"category": str(cat), "prev_val": round(vp, 2), "curr_val": round(vc, 2), "absolute_change": round(diff, 2), "pct_change": round((diff / vp * 100), 2) if vp > 0 else 0.0})
        seg_diffs.sort(key=lambda x: x["absolute_change"])
        breakdown[seg] = seg_diffs

    return {
        "metric_analyzed": m_col,
        "date_column_used": d_col,
        "previous_period": str(p_prev),
        "current_period": str(p_curr),
        "overall_performance": {"previous_total": round(v_prev, 2), "current_total": round(v_curr, 2), "absolute_delta": round(total_delta, 2), "relative_change_pct": pct_change},
        "breakdown_by_dimensions": breakdown
    }

ANOMALY_PROMPTS = {
    "v1": """Ти аналітик. Подивись на цю декомпозицію падіння показника між двома періодами і поясни, через що відбулося просідання.
Дані:
{anomaly_json}""",

    "v2": """Ти провідний продуктовий аналітик. Твоє завдання — пояснити причини падіння бізнес-метрики на основі факторного аналізу.
Вимоги:
1. Обов'язково посилайся на конкретні числа (періоди, відсоток загального падіння, збитки в абсолютних сумах).
2. Виділи топ-сегменти в кожному розрізі, які найбільше потягнули показник донизу.
3. Сформуй звіт:
   - Масштаб проблеми (загальні цифри)
   - Ключові сегменти-винуватці падіння
   - Ймовірні бізнес-причини
   - Рекомендовані термінові дії
Дані:
{anomaly_json}""",

    "v3": """<context>
Ти — Lead Financial / Product Analyst. Твоя мета — надати стейкхолдерам вичерпне, структуроване розслідування причин падіння метрики між двома періодами на основі розрахованої декомпозиції.
</context>
<rules>
1. Жодних вигаданих чисел: спирайся ТІЛЬКИ на <anomaly_data>.
2. Обов'язково виокреми основний сегмент-драйвер падіння у кожному доступному зрізі (канали, пристрої тощо).
3. Поверни строго валідний JSON зі зв'язними аналітичними формулюваннями.
</rules>
<anomaly_data>
{anomaly_json}
</anomaly_data>
<instructions>
Поверни JSON наступного формату:
{{
  "incident_summary": "Короткий огляд інциденту (2-3 речення): що впало, на скільки відсотків і в яких періодах.",
  "primary_drivers": [
    {{
      "dimension": "Назва виміру (наприклад: acquisition_channel)",
      "worst_segment": "Найбільш просілий сегмент",
      "metric_drop": "Абсолютна та відносна зміна",
      "share_of_loss": "Короткий аналіз впливу цього сегмента на загальний спад"
    }}
  ],
  "root_cause_hypotheses": [
    {{
      "hypothesis": "Ймовірна бізнес/технічна причина спаду в проблемному сегменті",
      "investigation_step": "Як швидко перевірити це припущення"
    }}
  ],
  "immediate_actions": "Зв'язний текст із 3-4 конкретними кроками для команди для стабілізації ситуації."
}}
</instructions>"""
}

def format_anomaly_v3(data):
    if not isinstance(data, dict) or "incident_summary" not in data: return str(data)
    lines = ["РОЗСЛІДУВАННЯ ПРОСІДАННЯ МЕТРИКИ:\n", f"ОГЛЯД СИТУАЦІЇ:\n{data.get('incident_summary', '')}\n"]
    if data.get("primary_drivers"):
        lines.append("ГОЛОВНІ ДРАЙВЕРИ ПАДІННЯ ПО ЗРІЗАХ:")
        for d in data["primary_drivers"]:
            lines.append(f" • Зріз '{d.get('dimension')}': сегмент '{d.get('worst_segment')}'")
            lines.append(f"   Зміна: {d.get('metric_drop')} ({d.get('share_of_loss')})")
        lines.append("")
    if data.get("root_cause_hypotheses"):
        lines.append("ПРИПУЩЕННЯ ЩОДО ПРИЧИН ТА ДІАГНОСТИКА:")
        for h in data["root_cause_hypotheses"]:
            lines.append(f" • Причина: {h.get('hypothesis')}")
            lines.append(f"   Дія для перевірки: {h.get('investigation_step')}")
        lines.append("")
    lines.append("НЕВІДКЛАДНІ ДІЇ:")
    lines.append(data.get("immediate_actions", ""))
    return "\n".join(lines)

def run_anomaly_explanation(df, prompt_version="v3", model="gemini-2.5-flash"):
    breakdown = extract_anomaly_breakdown(df)
    if "error" in breakdown: return breakdown["error"]
    p_json = json.dumps(breakdown, ensure_ascii=False, indent=2)
    prompt = ANOMALY_PROMPTS[prompt_version].format(anomaly_json=p_json)
    cfg = types.GenerateContentConfig(temperature=0.1, response_mime_type="application/json" if prompt_version == "v3" else "text/plain")
    res = client.models.generate_content(model=model, contents=prompt, config=cfg).text.strip()
    if prompt_version == "v3":
        try: return format_anomaly_v3(json.loads(res))
        except: return res
    return res

## Завдання 5. Формування Executive Summary для керівництва
Цей лок відповідає за підготовку стислого звіту. Програмний код збирає ключові макропоказники датасету в єдину зведену таблицю: загальний обсяг виручки за весь час, середній чек користувача, загальну кількість клієнтів, місячну динаміку, частку провідного каналу збуту та середній рівень задоволеності клієнтів.

Ці дані передаються штучному інтелекту. Модель формулює головний заголовок із ключовою цифрою, зв'язний огляд поточного стану бізнесу, розбирає стратегічні ризики і фіксує перелік першочергових завдань на наступний період.


In [7]:
def extract_executive_metrics(df):
    clean_df, mapping = map_dataframe_schema(df)
    m_col = mapping["metric_col"]
    d_col = mapping["date_cols"][-1] if mapping["date_cols"] else None

    total_records = int(len(clean_df))
    valid_metric = clean_df[m_col].dropna() if m_col else pd.Series([])
    total_vol = round(float(valid_metric.sum()), 2) if not valid_metric.empty else 0.0
    avg_val = round(float(valid_metric.mean()), 2) if not valid_metric.empty else 0.0

    period_change = None
    if d_col and not valid_metric.empty:
        df_d = clean_df.dropna(subset=[d_col, m_col]).copy()
        df_d['period'] = df_d[d_col].dt.to_period('M')
        periods = sorted(df_d['period'].unique())
        if len(periods) >= 2:
            p_prev, p_curr = periods[-2], periods[-1]
            vp = float(df_d[df_d['period'] == p_prev][m_col].sum())
            vc = float(df_d[df_d['period'] == p_curr][m_col].sum())
            pct = round(((vc - vp) / vp * 100), 2) if vp > 0 else 0.0
            period_change = {"current_period": str(p_curr), "prev_period": str(p_prev), "change_pct": pct}

    lead_segment = None
    if mapping["segments"] and not valid_metric.empty:
        primary_seg = mapping["segments"][0]
        grp = clean_df.groupby(primary_seg)[m_col].sum()
        if not grp.empty:
            lead_segment = {"dimension": primary_seg, "top_category": str(grp.idxmax()), "volume": round(float(grp.max()), 2), "share_pct": round(float(grp.max() / total_vol * 100), 2) if total_vol > 0 else 0.0}

    avg_score = None
    if mapping["rating_col"]:
        r_series = clean_df[mapping["rating_col"]].dropna()
        if not r_series.empty: avg_score = round(float(r_series.mean()), 2)

    return {
        "macro_metrics": {"analyzed_metric": m_col, "total_records": total_records, "total_volume": total_vol, "average_value": avg_val, "average_score": avg_score},
        "period_dynamics": period_change,
        "lead_segment": lead_segment,
        "discovered_dimensions": mapping["segments"][:4]
    }

SUMMARY_PROMPTS = {
    "v1": """Ти бізнес-аналітик. Подивись на зведені метрики продукту та напиши Executive Summary для генерального директора.
Дані:
{summary_json}""",

    "v2": """Ти Chief Product Officer (CPO). Твоє завдання — підготувати лаконічний executive-звіт для ради директорів за наданими цифрами.
Вимоги:
1. Посилайся виключно на конкретні розраховані цифри (виручка, динаміка місяця, частка каналів).
2. Жодних довгих вступів. Тільки сухі факти та стратегічні пріоритети.
3. Структура звіту:
   - Ключові результати (High-level Status)
   - Головні ризики та драйвери змін
   - Стратегічні кроки на наступний квартал
Показники:
{summary_json}""",

    "v3": """<context>
Ти — Chief Product Officer / VP of Analytics. Твоя мета — підготувати вичерпний, але лаконічний управлінський бриф (Executive Summary) для вищого керівництва на базі підрахованих агрегатів.
</context>
<rules>
1. Посилайся ТІЛЬКИ на конкретні факти з <kpi_data>. Заборонено вигадувати цифри.
2. Формулюй думки чіткою управлінською мовою: менше технічного шуму, більше бізнес-сенсу та грошей.
3. Відповідь має бути строго у форматі валідного JSON.
</rules>
<kpi_data>
{summary_json}
</kpi_data>
<instructions>
Сформуй JSON наступної структури:
{{
  "headline": "Гучний підсумок бізнес-стану компанії в один рядок (з ключовою цифрою)",
  "high_level_overview": "Зв'язний абзац (3-4 речення) про виручку, динаміку останнього місяця та стан клієнтської бази.",
  "strategic_highlights": [
    {{
      "pillar": "Канали росту | Ефективність промо | Задоволеність",
      "metric_fact": "Цитата точних даних із JSON",
      "business_takeaway": "Що цей факт означає для прибутковості"
    }}
  ],
  "strategic_action_plan": "Конкретний перелік із 3 ключових пріоритетів для команди на найближчий місяць."
}}
</instructions>"""
}

def format_summary_v3(data):
    if not isinstance(data, dict) or "headline" not in data: return str(data)
    lines = [f"EXECUTIVE SUMMARY: {data.get('headline', '')}\n", "ОГЛЯД СТАНУ БІЗНЕСУ:", f"{data.get('high_level_overview', '')}\n"]
    if data.get("strategic_highlights"):
        lines.append("КЛЮЧОВІ СТРАТЕГІЧНІ ФАКТИ:")
        for h in data["strategic_highlights"]:
            lines.append(f" • [{h.get('pillar')}]: {h.get('metric_fact')}")
            lines.append(f"   Бізнес-висновок: {h.get('business_takeaway')}")
        lines.append("")
    lines.append("СТРАТЕГІЧНІ ПРІОРИТЕТИ ТА ПЛАН ДІЙ:")
    lines.append(data.get("strategic_action_plan", ""))
    return "\n".join(lines)

def run_executive_summary(df, prompt_version="v3", model="gemini-2.5-flash"):
    kpis = extract_executive_metrics(df)
    p_json = json.dumps(kpis, ensure_ascii=False, indent=2)
    prompt = SUMMARY_PROMPTS[prompt_version].format(summary_json=p_json)
    cfg = types.GenerateContentConfig(temperature=0.1, response_mime_type="application/json" if prompt_version == "v3" else "text/plain")
    res = client.models.generate_content(model=model, contents=prompt, config=cfg).text.strip()
    if prompt_version == "v3":
        try: return format_summary_v3(json.loads(res))
        except: return res
    return res

### Тестування та порівняння варіантів промптів для первинного аудиту
Нижче послідовно запускаються три окремі виклики функції первинного аудиту. Перший виклик демонструє роботу базового запиту без чітких рамок, другий перевіряє роботу з рольовими обмеженнями, а третій повертає повну структуровану відповідь, перекладену у зв'язний текст. Кожен виклик винесено в окрему клітинку, щоб за потреби можна було змінити версію моделі або повторити запит без перезапуску всього коду.

In [ ]:
print(run_eda(df, prompt_version="v1", model="gemini-3.8-flash"))

Ось детальний звіт про якість наданого датасету (Data Quality Report), підготовлений на основі статистики та вибірки даних.

---

# 📊 Звіт про якість даних (Data Quality Report)

**Загальний обсяг даних:** 15 000 рядків, 30 колонок.  
**Повні дублікати:** 0 (0.0%).  
**Загальний стан:** Датасет має **високий рівень синтетичного шуму та логічних суперечностей**. Використовувати його для аналітики чи навчання моделей "як є" **неможливо** без попередньої глибокої очистки.

---

## 1. "Брудні" колонки та пропущені значення (Missing Values)

Суттєві пропуски спостерігаються в наступних колонках:

1. **`coupon_code`**: **40.89% пропусків** (6 133 рядки).
   * *Контекст:* Пропуски є нормальними, якщо користувач не застосовував промокод, але це суперечить прапорцю `discount_used` (див. Розділ 3).
2. **`age`**: **8.0% пропусків** (1 200 рядків).
3. **`total_spent`**: **7.0% пропусків** (1 050 рядків).
   * *Критичність:* Це критична бізнес-метрика. Її відсутність у 7% клієнтів заважає точній оц

In [ ]:
print(run_eda(df, prompt_version="v2", model="gemini-3.8-flash"))

Ось детальний аудит якості даних (Data Quality Audit Report), підготовлений для команди розробки, дата-інженерів та стейкхолдерів продукту.

---

# Звіт про якість даних (Data Quality Audit Report)

**Цільовий датасет:** Дані клієнтської поведінки та транзакцій  
**Обсяг:** 15,000 рядків | 30 стовпців | 0 дублікатів (0.0%)  
**Статус готовності до продакшн-аналітики / ML:** **Критичний (Blocked)**. Датасет містить ознаки синтетичної генерації або серйозних збоїв у пайплайнах збору даних (Data Integrity Breach).

---

## 1. Огляд стану датасету

Датасет є повнорозмірним профілем клієнтів (демографія, поведінка на сайті, фінанси, підтримка). На верхньому рівні відсутні повні дублікати рядків (`customer_id` унікальний для всіх 15,000 записів: від 10001 до 25000).

Проте детальний аудит виявив, що **дані в поточному вигляді не можна використовувати для прийняття продуктових рішень чи тренування ML-моделей (LTV, Churn)** через:
1. Системне порушення реляційної та бізнес-логіки між стовпцями

In [ ]:
print(run_eda(df, prompt_version="v3", model="gemini-3.8-flash"))

📊 ОЦІНКА ЗДОРОВ'Я ДАНИХ: 68/100

📌 ЗАГАЛЬНИЙ ВИСНОВОК:
Проведений аудит набору даних, що містить 15000 рядків та 30 колонок, виявив суттєві проблеми з цілісністю та якістю інформації, які наразі обмежують його пряме використання в аналітиці та побудові прогнозних моделей. У датасеті зафіксовано 0 дублікатів рядків, проте виявлено критичні хронологічні аномалії, коли дата останньої покупки передує даті реєстрації клієнта, а також системні географічні невідповідності між містами та країнами. Крім того, наявні пропуски у важливих фінансових і демографічних метриках, зокрема у колонках total_spent (1050 пропусків) та age (1200 пропусків із наявністю від'ємних значень). Для відновлення бізнес-придатності даних потрібна комплексна валідація логічних правил та очищення цільових полів перед розрахунком LTV і побудовою моделей відтоку.

⚠️ СУПЕРЕЧНОСТІ МІЖ СТОВПЦЯМИ (ЗНАЙДЕНО В СЕМПЛАХ):
 • [signup_date, last_purchase_date]: У вибірці виявлено пряме порушення часової послідовності: для customer

### Порівняльне тестування промптів для генерації гіпотез
У наступних клітинках ми окремо викликаємо три ітерації інструкцій для формування гіпотез. Це дозволяє наочно побачити, як додавання вимоги спиратися на конкретні числа змінює якість висновків моделі, перетворюючи загальні поради на практичний план дій для продуктової команди.

In [ ]:
print(run_hypotheses(df, prompt_version="v1", model="gemini-3.6-flash"))

Як продуктовий аналітик, я проаналізував агреговані дані та підготував **7 гіпотез** для покращення бізнес-показників (цільова метрика — `total_spent`). 

Нижче наведено гіпотези, розбиті за тематичними блоками, із зазначенням спостережень, суті гіпотези та рекомендованих дій.

---

### 1. Ефективність промокодів та канібалізація маржі
* **Спостереження:** Користувачі з промокодом `REF10` витрачають менше ($522.94), ніж користувачі без нього або з іншими кодами ($524.72). 
* **Гіпотеза 1:** Промокод `REF10` **канібалізує виторг**, а не стимулює додаткові покупки. Наявні високоінтенційні покупці використовують код просто для знижки на покупки, які вони зробили б і так.
* **Що перевірити / Action:**
  * Провести A/B тест із введенням **мінімального чека (AOV Threshold)** для активації коду `REF10` (наприклад, знижка діє тільки для замовлень від $600).
  * Порівняти метрику *LTV* та *CAC* для залучених через `REF10` користувачів.

---

### 2. Географічні аномалії: Лондон проти решти Велик

In [ ]:
print(run_hypotheses(df, prompt_version="v2", model="gemini-3.5-flash"))

Як провідний продуктовий аналітик, на основі наданих даних я сформулював 7 ключових продуктових та маркетингових гіпотез. Всі вони спрямовані на пошук точок зростання цільової метрики **`total_spent`**.

---

### Гіпотеза 1: Неефективність ціннісної пропозиції Преміум-підписки (Product/Monetization)
* **Спостереження (з цифрами):** Користувачі з преміум-статусом (`is_premium_user = 1`, count: 4,567) мають середній чек **$525.67** (медіана **$501.69**), тоді як не-преміум користувачі (`is_premium_user = 0`, count: 10,433) мають середній чек **$523.78** (медіана **$497.79**). Різниця в середніх витратах становить лише **$1.89** (менше 0.4%).
* **Припущення (чому так):** Поточний "Преміум" статус надає лише косметичні або нефункціональні фічі, які не стимулюють користувачів здійснювати додаткові або дорожчі покупки (upsell/cross-sell).
* **Що перевірити далі:** Провести аудит фічей Преміуму; розробити A/B тест з додаванням ексклюзивних знижок/безкоштовної експрес-доставки для Premium-кори

In [9]:
print(run_hypotheses(df, prompt_version="v3", model="gemini-3.5-flash"))

💡 ПРОДУКТОВІ ГІПОТЕЗИ ДЛЯ ПЕРЕВІРКИ:

Аналіз продуктових метрик виявив ключові точки втрати виручки у сегментах відтікаючих користувачів, клієнтів із проблемним саппорт-досвідом та критичними затримками логістики, а також потенціал для зростання LTV за рахунок оптимізації моделей підписки, регіонального ціноутворення та дисконтної політики.

📌 Гіпотеза №1: Своєчасне утримання ризикових користувачів запобіжить падінню LTV на 37.6%
 • Факт із даних: Витрати відтікаючих користувачів (churn = 1, count = 2298) становлять у середньому 347.79 (медіана 221.23) проти 557.29 (медіана 524.77) у збережених користувачів (churn = 0, count = 12702).
 • Припущення: Користувачі знижують транзакційну активність перед остаточним відтоком, що сигналізує про втрату зацікавленості або проблеми з цінностною пропозицією.
 • Як перевірити: Запустити предиктивну тригерну комунікацію (email/push) при зниженні частоти покупок і протестувати ретеншн-офер для користувачів з LTV нижче 300.

📌 Гіпотеза №2: Зменшення 

### Виклики різних версій аналізу аномалій


In [10]:
print(run_anomaly_explanation(df, prompt_version="v1", model="gemini-3.5-flash"))

Як аналітик, я провів аналіз наданої декомпозиції падіння загальних витрат (`total_spent`) між лютим (`2025-02`) та березнем (`2025-03`).

Ось детальний розбір причин, аномалій та гіпотез щодо того, чому відбулося просідання.

---

### 1. Загальний підсумок (Executive Summary)
* **Загальне падіння:** Виручка впала з **$251,701.45** до **$84,115.63**, що становить **-$167,585.82** або **-66.58%**.
* **Характер падіння:** Падіння є **системним та фронтальним** — показники впали приблизно на 60–70% практично за всіма зрізами (країни, міста, стать, канали). 
* **Ключовий драйвер аномального падіння:** Канал **Email**, який просів значно сильніше за інші (-80.08%).

---

### 2. Деталізований аналіз за вимірами

#### А. Канали залучення (Acquisition Channel) — *Найбільш інформативний зріз*
Усі канали просіли, але **Email** є головною аномалією:
1. **Email:** Втратив **-$40,896.39 (-80.08%)**. Це найбільше абсолютне та відносне падіння серед усіх каналів. На Email припадає **24.4% від усього 

In [ ]:
print(run_anomaly_explanation(df, prompt_version="v2", model="gemini-3.5-flash"))

Ось аналітичний звіт щодо результатів факторного аналізу падіння метрики `total_spent` за березень 2025 року.

---

# 📊 Аналітичний звіт: Факторний аналіз падіння `total_spent`

**Кому:** Leadership / Product Team  
**Автор:** Lead Product Analyst  
**Період аналізу:** Березень 2025 (`2025-03`) у порівнянні з Лютим 2025 (`2025-02`)  

---

## 1. Масштаб проблеми (Загальні цифри)

У березні 2025 року відбулося критичне падіння сумарного виторгу (`total_spent`):

* **Дохід у лютому (`2025-02`):** $251 701.45
* **Дохід у березні (`2025-03`):** $84 115.63
* **Абсолютні збитки:** **-$167 585.82**
* **Відносна зміна:** **-66.58%**

Бізнес втратив більше ніж **2/3 загального обсягу продажів** порівняно з попереднім місяцем. Падіння має системний характер, але виділяються конкретні сегменти, які зазнали найбільших втрат.

---

## 2. Ключові сегменти-винуватці падіння

Аналіз у розрізі чотирьох вимірів (Dimensions) дозволив визначити головні точки просідання:

### ✉️ Канали залучення (Acquisiti

In [ ]:
print(run_anomaly_explanation(df, prompt_version="v3", model="gemini-3.5-flash"))

📉 РОЗСЛІДУВАННЯ ПРОСІДАННЯ МЕТРИКИ:

📌 ОГЛЯД СИТУАЦІЇ:
У періоді 2025-03 зафіксовано критичне падіння загальних витрат користувачів (total_spent) на 66.58% порівняно з 2025-02. Загальний обсяг метрики зменшився з 251701.45 до 84115.63, що призвело до абсолютних втрат у розмірі -167585.82.

🔍 ГОЛОВНІ ДРАЙВЕРИ ПАДІННЯ ПО ЗРІЗАХ:
 • Зріз 'acquisition_channel': сегмент 'Email'
   Зміна: -40896.39 (-80.08%) (Канал Email показав найбільше абсолютне та відносне падіння серед усіх каналів залучення, сформувавши значну частку загального спаду метрики.)
 • Зріз 'device_type': сегмент 'Desktop'
   Зміна: -60398.62 (-68.97%) (Десктопні пристрої стали основним драйвером падіння у зрізі пристроїв, зменшившись з 87576.34 до 27177.73.)
 • Зріз 'gender': сегмент 'Female'
   Зміна: -81107.85 (-71.29%) (Сегмент Female зазнав найбільших втрат серед демографічних груп, згенерувавши майже половину загального абсолютного падіння.)
 • Зріз 'country': сегмент 'Germany'
   Зміна: -35656.07 (-66.82%) (Німеччина 

### Генерація звіту за трьома варіантами промптів


In [ ]:
print(run_executive_summary(df, prompt_version="v1", model="gemini-3.5-flash"))

Ось аналітична записка (Executive Summary), складена за стандартами ділової листування для вищого керівництва (C-level).

---

# EXECUTIVE SUMMARY: Аналіз продуктових метрик та динаміки продажів
**Кому:** Генеральному директору (CEO)  
**Від:** Бізнес-аналітика  
**Звітний період:** Березень 2025 року  

---

### 🔴 Головний висновок (BLUF — Bottom Line Up Front)
У березні 2025 року зафіксовано **критичне падіння виручки на 66.58%** порівняно з лютим. Попри формування солідної клієнтської бази (15 000 користувачів) та високий середній чек ($524.36), бізнес демонструє **небезпечну залежність від промо-активностей (пронікання купонів 59.1%)** та помірний рівень задоволеності клієнтів (CSAT 3.59/5), що створює високі ризики для утримання доходів у довгостроковій перспективі.

---

### 📊 Ключові показники (Macro Overview)

* **Загальна виручка:** $7,314,785.69
* **Клієнтська база:** 15,000 унікальних клієнтів
* **Середній чек (AOV):** $524.36 *(сильний показник, що вказує на платоспроможну 

In [ ]:
print(run_executive_summary(df, prompt_version="v2", model="gemini-3.5-flash"))

**Звіт для Ради директорів: Продукт та Бізнес-показники**

### 1. Ключові результати (High-level Status)
* **Загальна виручка:** 7 314 785,69
* **Клієнтська база:** 15 000 користувачів
* **Середній чек (AOV):** 524,36
* **Рівень задоволеності (CSAT):** 3,59
* **Місячна динаміка (MoM):** Падіння виручки на **-66,58%** у березні 2025 року відносно лютого 2025 року.
* **Ключовий канал продажів:** Organic — генерує **20,67%** загальної виручки (1 511 907,97).

---

### 2. Головні ризики та драйвери змін
* **Критичне падіння доходів (-66,58% MoM):** Різке скорочення виручки у березні 2025 р. вимагає негайного операційного та продуктового втручання.
* **Залежність від промо-активностей (59,11%):** Висока проникність купонів свідчить про чутливість аудиторії до знижок та ризик падіння маржинальності продажів.
* **Низька задоволеність продуктом (CSAT 3,59):** Показник сигналізує про проблеми з утриманням (retention) 15 000 наявних клієнтів та якістю продуктового досвіду.
* **Концентрація Organ

In [ ]:
print(run_executive_summary(df, prompt_version="v3", model="gemini-3.6-flash"))

📋 EXECUTIVE SUMMARY: Критичне падіння доходу на 66.58% у березні вимагає негайної оптимізації промо-маржи та стабілізації продуктів

📌 ОГЛЯД СТАНУ БІЗНЕСУ:
Загальний дохід компанії за весь період склав 7 314 785.69 грн при загальній клієнтській базі в 15 000 користувачів та середньому чеку 524.36 грн. Проте в березні 2025 року зафіксовано різке падіння виручки на 66.58% порівняно з лютим 2025 року. Ситуація ускладнюється помірним показником задоволеності клієнтів, який становить лише 3.59 бала.

🎯 КЛЮЧОВІ СТРАТЕГІЧНІ ФАКТИ:
 • [Канали росту]: Органічний канал (Organic) приніс 1 511 907.97 грн виручки, що становить 20.67% від загального обсягу
   Бізнес-висновок: Органіка є ключовим драйвером продажів, проте компанії необхідно диверсифікувати джерела залучення для зменшення ризиків падіння загального доходу.
 • [Ефективність промо]: Пронікання купонів становить 59.11%
   Бізнес-висновок: Високий рівень промо-залежності каннібалізує маржинальність та привчає базу купувати виключно зі зни